In [ ]:
# ===== STABLE-AC CoV (case i) — CONFIG (edit ONLY this cell) ==============

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "test/stable-ac-moves-w4"   # must match the actual git branch
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push

MOUNT_DRIVE = True           # mirror results/stable_ac/cov/*.jsonl to Drive every
                             # few minutes + at the end, and seed them BACK on a
                             # fresh VM so resume continues where the last session
                             # stopped. The runner always writes locally (appending
                             # straight onto the Drive FUSE mount is slow/flaky).
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx_results/stable_ac_cov"

# --- experiment knobs ------------------------------------------------------
# Defaults live in experiments/stable_ac/cov/config_cov.yaml (the reviewed
# file); anything set here overrides it. One resumable jsonl per budget.
CONFIG_PATH = "experiments/stable_ac/cov/config_cov.yaml"
BUDGET = [100, 1000]         # local proof budgets; production (e.g. [50000]) runs here on Colab
MODE   = "cov"               # "cov" | "baseline" (identity transform -> comparison file)
EXPERIMENT_LENGTH = False    # True = length sweep: greedy from EVERY valid CoV + a no-CoV
                             # control row per presentation (mode must be "cov");
                             # prints the length-vs-outcome digest at the end
Z_SOURCE = None              # None = yaml default ("subwords" = z from the relators).
                             # "universe" = every reduced word of len 2..universe_max_len
                             # (z need not occur — Nielsen re-coordinatisations; sweep only)
DATASETS = None              # None = the yaml's local proof pair (subset_10 + reach_1).
                             # Production 66-row benchmark:
                             # DATASETS = ["results/benchmark/subsets/benchmark_subset_60.csv",
                             #             "results/benchmark/reach/reach_tier_6.csv"]

HIGH_SPEEDUP = True          # compact fast solver (~2.9x); result-neutral — every
                             # written row is identical to a slow-mode row (solved
                             # rows are re-solved by the normal solver for the path),
                             # and files resume across the two modes. Overrides the
                             # yaml; set False to force the plain solver.

USE_CHUNKS  = False          # True = stride-split the presentations into CHUNKS
                             # chunks; each chunk writes its own _c{i}of{N}_ jsonl
CHUNKS      = 3
CHUNK_INDEX = None           # None = run ALL chunks as parallel processes in THIS
                             # session (high-RAM multi-vCPU runtime); 1..CHUNKS =
                             # run only that chunk (one per parallel Colab session);
                             # a LIST like [1, 5, 9] = run those chunks as parallel
                             # processes here (split a finer partition across
                             # several sessions)
OLD_CHUNKS  = None           # set to the PREVIOUS partition size (e.g. 4) when
                             # re-partitioning a live sweep to a finer CHUNKS:
                             # RUN first re-bins every finished row from the old
                             # _c{i}of{OLD_CHUNKS}_ files into the new partition
                             # (idempotent, nothing re-runs), then searches


In [ ]:
# ==================== SETUP (clone / pull / install) ======================
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy pyyaml")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

# run from repo root so relative paths and "import experiments..." resolve
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# a `git reset --hard` rewrites .py files but sys.modules keeps the OLD module
# objects -- drop them so RUN imports what SETUP just fetched (pull != reload)
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()


# --- Drive: mount + seed-back (fresh VM -> local resume state) ------------
import glob, shutil
LOCAL_OUT = os.path.join(REPO_ROOT, "results", "stable_ac", "cov")
os.makedirs(LOCAL_OUT, exist_ok=True)
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for src in glob.glob(os.path.join(DRIVE_DIR, "*.jsonl")):
        dst = os.path.join(LOCAL_OUT, os.path.basename(src))
        # the bigger file wins: local mid-run state beats a stale mirror,
        # and on a fresh VM the mirror beats the (absent/empty) local file
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            shutil.copyfile(src, dst)
            print("seeded from Drive:", os.path.basename(src))


In [ ]:
# ==================== RUN =================================================
from experiments.stable_ac.cov.run_cov import run

# mirror local jsonls -> Drive every 3 min (and once at the end). Whole-file
# copies of append-only jsonls: a torn tail line is repaired on resume. The
# thread never prints (background threads must not).
import threading, time as _time
def _sync_to_drive():
    if not (IN_COLAB and MOUNT_DRIVE):
        return
    for src in glob.glob(os.path.join(LOCAL_OUT, "*.jsonl")):
        dst = os.path.join(DRIVE_DIR, os.path.basename(src))
        # size-monotonic: an append-only jsonl only ever grows, so never
        # overwrite a bigger Drive copy with a smaller local one (a session
        # holding a seeded STALE copy of another session's chunk must not
        # clobber the owner's fresh mirror)
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            tmp = dst + ".tmp"
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
def _mirror_loop():
    while not _mirror_stop.wait(180):
        try: _sync_to_drive()
        except Exception: pass                 # transient Drive hiccup: next tick
_mirror_stop = threading.Event()
threading.Thread(target=_mirror_loop, daemon=True).start()

if USE_CHUNKS and OLD_CHUNKS:
    from experiments.stable_ac.cov.run_cov import rechunk
    rechunk(config_path=CONFIG_PATH, budgets=BUDGET, mode=MODE,
            experiment_length=EXPERIMENT_LENGTH, z_source=Z_SOURCE,
            datasets=DATASETS, use_chunks=True, chunks=CHUNKS,
            old_chunks=OLD_CHUNKS)

run(config_path=CONFIG_PATH, budgets=BUDGET, mode=MODE,   # resumable; one jsonl per budget
    experiment_length=EXPERIMENT_LENGTH, z_source=Z_SOURCE, datasets=DATASETS,
    high_speedup=HIGH_SPEEDUP,
    use_chunks=USE_CHUNKS, chunks=CHUNKS, chunk_index=CHUNK_INDEX)

_mirror_stop.set()
_sync_to_drive()                               # final sync, incl. the last rows
if IN_COLAB and MOUNT_DRIVE:
    print("mirrored to", DRIVE_DIR)


In [ ]:
# ==================== MERGE (chunked runs only, after ALL chunks finish) ==
# Concatenates the completed _c{i}of{N}_ chunk jsonls into the canonical
# unchunked file. It REFUSES to run while any chunk is missing rows (counts are
# re-derived by enumeration) and when the target already exists, so running it
# early or twice is safe. Verify certificates afterwards:
#   .venv/bin/python3 -m experiments.stable_ac.verify_results results/stable_ac/cov
if USE_CHUNKS:
    from experiments.stable_ac.cov.run_cov import merge_chunks
    merge_chunks(config_path=CONFIG_PATH, budgets=BUDGET, mode=MODE,
                 experiment_length=EXPERIMENT_LENGTH, z_source=Z_SOURCE,
                 datasets=DATASETS, use_chunks=True, chunks=CHUNKS)
    _sync_to_drive()                           # mirror the merged file too
